<a href="https://colab.research.google.com/github/essanchristian-maker/DI-Bootcamp/blob/master/Week7_Day3_Gold.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers

In [ ]:
from transformers import AutoTokenizer, AutoModel

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained("bert-base-uncased")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
# On déplace les inputs sur le même appareil (GPU ou CPU) que le modèle
inputs = tokenizer("Hello Hugging Face!", return_tensors="pt").to(model.device)
outputs = model(**inputs)
print("Calcul réussi !")

Calcul réussi !


In [ ]:
!pip install -qU langchain langchain-community langchain-huggingface datasets faiss-cpu sentence-transformers pyarrow

import langchain
print(f"LangChain version: {langchain.__version__}")
print("✅ Installation terminée. Si c'est la première fois, REDÉMARREZ LA SESSION maintenant.")

LangChain version: 1.3.11
✅ Installation terminée. Si c'est la première fois, REDÉMARREZ LA SESSION maintenant.


In [ ]:
pip install -Uq datasets

In [ ]:
dataset_name = "databricks/databricks-dolly-15k"
page_content_column = "context"

In [ ]:
import os
from langchain_core.documents import Document

try:
    from datasets import load_dataset
    ds = load_dataset(dataset_name, split='train', trust_remote_code=True)
    data = [Document(page_content=str(x[page_content_column]), metadata=x) for x in ds.select(range(min(500, len(ds))))]
    print(f"Succès : {len(data)} documents chargés.")
except Exception as e:
    print(f"Erreur pyarrow persistante : {e}")
    print("CONSEIL : Redémarrez la session via le menu 'Exécution' puis relancez tout.")

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'databricks/databricks-dolly-15k' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'databricks/databricks-dolly-15k' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Succès : 500 documents chargés.


In [ ]:
# Reinstall pyarrow and datasets to resolve potential binary incompatibility issues
!pip install -Uq pyarrow
!pip install -Uq datasets

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)

In [ ]:
# Aggressive reinstallation to resolve persistent pyarrow binary incompatibility
!pip uninstall -y pyarrow datasets sentence-transformers
!pip install -Uq pyarrow datasets sentence-transformers

Found existing installation: pyarrow 24.0.0
Uninstalling pyarrow-24.0.0:
  Successfully uninstalled pyarrow-24.0.0
Found existing installation: datasets 5.0.0
Uninstalling datasets-5.0.0:
  Successfully uninstalled datasets-5.0.0
Found existing installation: sentence-transformers 5.6.0
Uninstalling sentence-transformers-5.6.0:
  Successfully uninstalled sentence-transformers-5.6.0


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
docs = text_splitter.split_documents(data)
print(f"Découpé en {len(docs)} segments (chunks)")

Découpé en 209 segments (chunks)


In [ ]:
modelPath = "sentence-transformers/all-MiniLM-l6-v2"
model_kwargs = {'device':'cpu'}
encode_kwargs = {'normalize_embeddings': False}

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
  model_name=modelPath,
  model_kwargs=model_kwargs,
  encode_kwargs=encode_kwargs
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
# Install the recommended langchain-huggingface package
!pip install -U langchain-huggingface


In [ ]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
  model_name=modelPath,
  model_kwargs=model_kwargs,
  encode_kwargs=encode_kwargs
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
text = "This is a test document."
query_result = embeddings.embed_query(text)
print(query_result[:3])

[-0.03833857178688049, 0.12346471846103668, -0.028642943128943443]


In [ ]:
try:
    from langchain_community.vectorstores import FAISS
except ImportError:
    !pip install -U langchain-community
    from langchain_community.vectorstores import FAISS

db = FAISS.from_documents(docs, embeddings)

6. Préparation du modèle LLM : Le modèle de langage est chargé de générer des réponses à partir des documents extraits. Le chargement d’un modèle pré-entraîné et son intégration dans un pipeline Langchain facilitent son intégration au système de recherche.

In [ ]:
from transformers import AutoTokenizer, AutoModelForQuestionAnswering, pipeline
from langchain_huggingface.llms import HuggingFacePipeline

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("Intel/dynamic_tinybert")
model = AutoModelForQuestionAnswering.from_pretrained("Intel/dynamic_tinybert")

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/351 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/268M [00:00<?, ?B/s]

Invalid model-index. Not loading eval results into CardData.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[transformers] BertForQuestionAnswering LOAD REPORT from: Intel/dynamic_tinybert
Key              | Status     |  | 
-----------------+------------+--+-
fit_dense.weight | UNEXPECTED |  | 
fit_dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer

# Utilisation d'un modèle de génération de texte plus stable pour cet environnement
model_id = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id)

# On utilise 'text-generation' qui est listé comme disponible dans votre erreur
qa_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=50
)

print("Pipeline de génération prêt. Si cette cellule échoue, redémarrez la session.")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Pipeline de génération prêt. Si cette cellule échoue, redémarrez la session.


In [ ]:
from langchain_huggingface import HuggingFacePipeline
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer

model_id = "gpt2"
tok = AutoTokenizer.from_pretrained(model_id)
mod = AutoModelForCausalLM.from_pretrained(model_id)
gen_pipeline = pipeline("text-generation", model=mod, tokenizer=tok, max_new_tokens=50)

llm = HuggingFacePipeline(pipeline=gen_pipeline)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

7. Création de la chaîne de questions-réponses pour la recherche : Cette chaîne relie le module de recherche (qui trouve les documents pertinents) au module de gestion des réponses (qui génère les réponses). Elle permet le déroulement complet du processus RAG (Retrieve, Agility, Answer), où le système récupère le contexte utile puis répond à la requête de l’utilisateur en fonction de ce contexte.

In [ ]:
retriever = db.as_retriever(search_kwargs={"k": 4}) # Optional: You can adjust k for number of documents retrieved

In [ ]:
import sys
import os

try:
    from langchain.chains import RetrievalQA
    from langchain_huggingface import HuggingFacePipeline

    if 'db' in globals() and 'llm' in globals():
        retriever = db.as_retriever(search_kwargs={"k": 3})
        qa = RetrievalQA.from_chain_type(
            llm=llm,
            chain_type="stuff",
            retriever=retriever
        )
        print("✅ Système RAG prêt !")

        # Test rapide
        query = "When did Virgin Australia start operating?"
        print(f"\nTest de requête : {query}")
        print(qa.invoke(query)['result'])
    else:
        print("❌ Erreur : 'db' ou 'llm' non définis. Assurez-vous d'avoir exécuté les cellules précédentes après le redémarrage.")
except ModuleNotFoundError:
    print("❌ LangChain n'est toujours pas détecté.")
    print("👉 SOLUTION : Menu 'Exécution' -> 'Redémarrer la session', puis relancez tout.")
except Exception as e:
    print(f"❌ Une erreur est survenue : {e}")

❌ LangChain n'est toujours pas détecté.
👉 SOLUTION : Menu 'Exécution' -> 'Redémarrer la session', puis relancez tout.


8. Testez votre système RAG : L’exécution d’une requête de test vous permet de vérifier que tous les composants fonctionnent correctement ensemble. Cette étape garantit que les documents sont récupérés correctement et que le modèle génère des réponses pertinentes en fonction du contexte récupéré.